In [1]:
from pathlib import Path
import duckdb
import pandas as pd

class DataHubReader:
    def __init__(self, db_name="yahoo_finance.db"):
        """
        Stellt Verbindung zur DuckDB her.
        Erwartet dieselbe Ordnerstruktur wie DataHub.
        """
        base_path = Path.cwd()
        db_path = base_path.parent.parent / "data" / "01_raw" / "yahoo" / db_name

        if not db_path.exists():
            raise FileNotFoundError(f"Datenbank nicht gefunden: {db_path}")

        self.con = duckdb.connect(str(db_path))

    def _fetch_table(self, table_name: str) -> pd.DataFrame:
        """Generische Methode zum Laden einer Tabelle."""
        return self.con.execute(f"SELECT * FROM {table_name}").df()

    def get_financials(self) -> pd.DataFrame:
        """Yahoo Finance Financials"""
        return self._fetch_table("bronze_financials")

    def get_wikidata(self) -> pd.DataFrame:
        """Wikidata Unternehmensdaten"""
        return self._fetch_table("bronze_wikidata")

    def get_GMD(self) -> pd.DataFrame:
        """Global Macro Database"""
        return self._fetch_table("bronze_gmd")

    def get_interest(self) -> pd.DataFrame:
        """IMF Zinsdaten"""
        return self._fetch_table("bronze_interest")

    def get_commodity(self) -> pd.DataFrame:
        """IMF Commodity Daten"""
        return self._fetch_table("bronze_commodities")

    def close(self):
        """Schließt die Datenbankverbindung."""
        self.con.close()

In [4]:
class GMDtoSilver():
    def __init__(self):
        self.comparable = ['nGDP_USD', 'rGDP_USD', 'deflator', 'cons_GDP', 'cons_USD', 'inv_GDP', 'inv_USD', 'finv_GDP', 'finv_USD','exports_GDP','exports_USD', 
        'imports_GDP', 'imports_USD','CA_GDP', 'REER', 'USDfx','gen_govexp_GDP','cgovexp_GDP', 'gen_govrev_GDP', 'cgovrev_GDP','gen_govtax_GDP', 
        'cgovtax_GDP', 'govdef_GDP', 'gen_govdef_GDP', 'cgovdef_GDP','govdebt_GDP', 'gen_govdebt_GDP','cgovdebt_GDP','HPI', 'CPI', 'infl','pop', 
        'unemp','strate', 'ltrate', 'cbrate','SovDebtCrisis', 'CurrencyCrisis', 'BankingCrisis','CA_USD', 'govexp_GDP', 'govrev_GDP', 'govtax_GDP', 
        'rGDP_pc_USD', 'income_group_code']
    
    def filter_comparable(self,df):
        return df[df['item_description'].isin(self.comparable)]
    
    def filter_recent_historical_data(self, df, up_bound=2026, low_bound=2000):
        return df[(df['year'] > low_bound) & (df['year'] < up_bound)]
    def unit_naming(self, df):
        def map_unit(item):
            if not isinstance(item, str):
                print("ITEMDESCRIPTION WAR KEIN STRING")
                return None
            
            item = item.strip()
            
            rules = [
                ("USD", "USD"),
                ("GDP", "Percent"),
                ("deflator", "Index"),
                ("REER", "Index"),
                ("USDfx", "Multiplicator"),
                ("HPI", "Index"),
                ("CPI", "Index"),
                ("infl", "Percent"),
                ("pop", "Amount"),
                ("unemp", "Percent"),
                ("strate", "Percent"),
                ("ltrate", "Percent"),
                ("cbrate", "Percent"),
                ("Crisis", "Rating"),
                ("income_group_code", "Rating")

            ]
            
            for suffix, unit in rules:
                if item.endswith(suffix):
                    return unit
            
            return None

        df = df.copy()
        df["unit"] = df["item_description"].apply(map_unit)
        return df
    def affiliation(self, df):
        category_mapping = {
            "nGDP_USD": "national_accounts",
            "rGDP_USD": "national_accounts",
            "deflator": "national_accounts",
            "cons_GDP": "national_accounts",
            "cons_USD": "national_accounts",
            "inv_GDP": "national_accounts",
            "inv_USD": "national_accounts",
            "finv_GDP": "national_accounts",
            "finv_USD": "national_accounts",
            "rGDP_pc_USD": "national_accounts",
            "exports_GDP": "external_sector",
            "exports_USD": "external_sector",
            "imports_GDP": "external_sector",
            "imports_USD": "external_sector",
            "CA_GDP": "external_sector",
            "CA_USD": "external_sector",
            "REER": "external_sector",
            "USDfx": "external_sector",
            "gen_govexp_GDP": "government_finance",
            "cgovexp_GDP": "government_finance",
            "govexp_GDP": "government_finance",
            "gen_govrev_GDP": "government_finance",
            "cgovrev_GDP": "government_finance",
            "govrev_GDP": "government_finance",
            "gen_govtax_GDP": "government_finance",
            "cgovtax_GDP": "government_finance",
            "govtax_GDP": "government_finance",
            "govdef_GDP": "government_finance",
            "gen_govdef_GDP": "government_finance",
            "cgovdef_GDP": "government_finance",
            "govdebt_GDP": "government_finance",
            "gen_govdebt_GDP": "government_finance",
            "cgovdebt_GDP": "government_finance",
            "HPI": "prices",
            "CPI": "prices",
            "infl": "prices",
            "unemp": "labor_market",
            "strate": "monetary_policy",
            "ltrate": "monetary_policy",
            "cbrate": "monetary_policy",
            "pop": "population",
            "SovDebtCrisis": "crisis",
            "CurrencyCrisis": "crisis",
            "BankingCrisis": "crisis",
            "income_group_code": "classification",
        }

        df = df.copy()
        df["affiliation"] = df["item_description"].map(category_mapping).fillna("other")
        return df

    def source_naming(self, df):
        dff = df.copy()
        dff['source'] = 'GlobalMacroData'
        return dff
    
    def run(self, df):
        df = df.copy()
        comp = self.filter_comparable(df)
        current = self.filter_recent_historical_data(comp)
        source = self.source_naming(current)
        affiliation = self.affiliation(source)
        units = self.unit_naming(affiliation)
        units['EntityType'] = 'States'
        return units

gmd = DataHubReader().get_GMD()
gmd_after = GMDtoSilver().run(gmd)
gmd_after.to_csv("gmd.csv", encoding="utf-8")

In [3]:
gmd_after

,countryname,iso3,year,item_description,value,ingested_at,source,affiliation,unit,EntityType
56890,Aruba,ABW,2001,nGDP_USD,1896.457,2026-03-11 08:49:18.152004,GlobalMacroData,national_accounts,USD,States
56891,Aruba,ABW,2002,nGDP_USD,1961.844,2026-03-11 08:49:18.152004,GlobalMacroData,national_accounts,USD,States
56892,Aruba,ABW,2003,nGDP_USD,2044.112,2026-03-11 08:49:18.152004,GlobalMacroData,national_accounts,USD,States
56893,Aruba,ABW,2004,nGDP_USD,2254.831,2026-03-11 08:49:18.152004,GlobalMacroData,national_accounts,USD,States
56894,Aruba,ABW,2005,nGDP_USD,2360.017,2026-03-11 08:49:18.152004,GlobalMacroData,national_accounts,USD,States
...,...,...,...,...,...,...,...,...,...,...
4206816,Zimbabwe,ZWE,2021,income_group_code,2.000,2026-03-11 08:49:18.152004,GlobalMacroData,classification,Rating,States
4206817,Zimbabwe,ZWE,2022,income_group_code,2.000,2026-03-11 08:49:18.152004,GlobalMacroData,classification,Rating,States
4206818,Zimbabwe,ZWE,2023,income_group_code,2.000,2026-03-11 08:49:18.152004,GlobalMacroData,classification,Rating,States
4206819,Zimbabwe,ZWE,2024,income_group_code,2.000,2026-03-11 08:49:18.152004,GlobalMacroData,classification,Rating,States
